# Experiment No. 1 — Data Visualization Task

**Aim:** To explore a dataset and create appropriate visualizations for understanding patterns, distributions, trends, and relationships in the data.

**Dataset:** Credit Card Details (Kaggle): https://www.kaggle.com/datasets/rohitudageri/credit-card-details  
Each row is one credit card applicant. `label`: **0 = Approved, 1 = Rejected**.

*Detailed explanations of each step are in the experiment report (Word document).*

## Task 1 — Load and Inspect the Dataset

In [ ]:
# Import the required libraries
import os                          # to build file paths
import pandas as pd                # to load and handle the data table
import matplotlib.pyplot as plt    # to draw charts
import seaborn as sns              # to draw statistical charts easily

# Use a white background with grid lines for all charts
sns.set_theme(style='whitegrid')

Run **only one** of the next two cells: Option 1 on a laptop, Option 2 on Google Colab.

In [ ]:
# OPTION 1 - Running on laptop (CSV files are in the same folder as this notebook)
path = '.'    # '.' means the current folder

In [ ]:
# OPTION 2 - Running on Google Colab (download the dataset from Kaggle)
# Remove the # from the two lines below when using Colab.

# import kagglehub
# path = kagglehub.dataset_download('rohitudageri/credit-card-details')

In [ ]:
# Read both CSV files into pandas DataFrames (tables)
df_details = pd.read_csv(os.path.join(path, 'Credit_card.csv'))        # applicant details
df_label = pd.read_csv(os.path.join(path, 'Credit_card_label.csv'))    # approved / rejected result

# shape gives (number of rows, number of columns)
print('Credit_card.csv shape       :', df_details.shape)
print('Credit_card_label.csv shape :', df_label.shape)

In [ ]:
# Join the two tables using the common column Ind_ID
df = pd.merge(df_details, df_label, on='Ind_ID')

print('Merged dataset shape:', df.shape)

# Show the first 5 rows to see what the data looks like
df.head()

In [ ]:
# info() shows each column's name, how many values are filled (non-null), and its data type
df.info()

**Observation:** 1,548 rows and 19 columns. Some columns have fewer than 1,548 filled values, so they have missing values.

## Task 2 — Identify Numerical and Categorical Attributes

In [ ]:
# Data type of every column
# int64 / float64 = numbers,  object = text
df.dtypes

In [ ]:
# Pick columns stored as numbers and columns stored as text
numerical_cols = df.select_dtypes(include='number').columns.tolist()
categorical_cols = df.select_dtypes(include='object').columns.tolist()

# Ind_ID is only an ID for each person, not a real number to analyse, so remove it
numerical_cols.remove('Ind_ID')

print('Numerical columns  :', numerical_cols)
print()
print('Categorical columns:', categorical_cols)

**Note:** `Mobile_phone`, `Work_Phone`, `Phone`, `EMAIL_ID`, and `label` are stored as 0/1 numbers, but they are really Yes/No **categories**.

## Task 3 — Basic Pre-processing

In [ ]:
# duplicated() marks rows that are exact copies of an earlier row; sum() counts them
print('Number of duplicate rows:', df.duplicated().sum())

In [ ]:
# Count the missing (empty) values in each column
missing = df.isnull().sum()

# Keep only the columns that actually have missing values
missing = missing[missing > 0]

# Make a small table with the count and the percentage of missing values
missing_table = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': (missing / len(df) * 100).round(2)
})
missing_table

In [ ]:
# Compare the mean and median of income to decide how to fill missing income values
print('Mean income  :', round(df['Annual_income'].mean()))
print('Median income:', df['Annual_income'].median())

The mean is higher than the median because of a few very high incomes, so missing numbers are filled with the **median**. Missing text values are filled with `'Unknown'`.

In [ ]:
# Fill missing text values with the word 'Unknown'
df['Type_Occupation'] = df['Type_Occupation'].fillna('Unknown')
df['GENDER'] = df['GENDER'].fillna('Unknown')

# Fill missing number values with the median (middle value) of that column
df['Annual_income'] = df['Annual_income'].fillna(df['Annual_income'].median())
df['Birthday_count'] = df['Birthday_count'].fillna(df['Birthday_count'].median())

# Check again: this should now print 0
print('Total missing values now:', df.isnull().sum().sum())

In [ ]:
# Convert age from negative days to years (e.g. -18772 days -> 51 years)
df['Age'] = (-df['Birthday_count'] / 365).round().astype(int)

# Replace 0/1 with readable words for the charts
df['Status'] = df['label'].map({0: 'Approved', 1: 'Rejected'})

# Show the old and new columns side by side
df[['Birthday_count', 'Age', 'label', 'Status']].head()

## Tasks 4, 5, 6 — Visualizations

### Visualization 1 — Approved vs Rejected (Count Plot)
**Why:** `Status` is categorical, and a count plot compares the size of each group.

In [ ]:
plt.figure(figsize=(7, 5))    # set the chart size (width, height)

# countplot counts the rows in each group and draws one bar per group
ax = sns.countplot(data=df, x='Status', order=['Approved', 'Rejected'])

# Write the count on top of each bar
ax.bar_label(ax.containers[0])

plt.title('Approved vs Rejected Credit Card Applications')
plt.xlabel('Application Status')
plt.ylabel('Number of Applications')
plt.show()

# Percentage of each group
print((df['Status'].value_counts(normalize=True) * 100).round(1))

**Observation:** 1,373 approved (88.7%) and 175 rejected (11.3%), so the data is **imbalanced**.

### Visualization 2 — Annual Income (Histogram)
**Why:** `Annual_income` is numerical, and a histogram shows how its values are spread.

In [ ]:
plt.figure(figsize=(9, 5))

# bins=30 -> split the income range into 30 bars
# kde=True -> also draw a smooth curve showing the overall shape
sns.histplot(data=df, x='Annual_income', bins=30, kde=True)

plt.title('Distribution of Annual Income')
plt.xlabel('Annual Income')
plt.ylabel('Number of Applicants')
plt.show()

**Observation:** Most incomes are between 1 and 3 lakh, with a long tail of a few very high incomes. The distribution is **right-skewed**.

### Visualization 3 — Annual Income (Box Plot)
**Why:** A box plot shows the median, the spread, and the **outliers** in one picture.

In [ ]:
plt.figure(figsize=(9, 3.5))

# Horizontal box plot of income
sns.boxplot(x=df['Annual_income'])

plt.title('Box Plot of Annual Income')
plt.xlabel('Annual Income')
plt.show()

In [ ]:
# Count the outliers using the IQR rule
Q1 = df['Annual_income'].quantile(0.25)    # 25% of people earn less than this
Q3 = df['Annual_income'].quantile(0.75)    # 75% of people earn less than this
IQR = Q3 - Q1                              # width of the box
upper_limit = Q3 + 1.5 * IQR               # anything above this is an outlier

# Select the rows whose income is above the upper limit
outliers = df[df['Annual_income'] > upper_limit]

print('Q1          :', Q1)
print('Q3          :', Q3)
print('IQR         :', IQR)
print('Upper limit :', upper_limit)
print('Number of high-income outliers:', len(outliers))

**Observation:** **73 applicants** earn more than the upper limit of 3,80,250. They are kept, because they are genuine high earners and not errors.

### Visualization 4 — Education Levels (Horizontal Count Plot)
**Why:** `EDUCATION` is categorical. The bars are horizontal because the category names are long.

In [ ]:
plt.figure(figsize=(10, 5))

# Order the categories from most common to least common
order = df['EDUCATION'].value_counts().index

# y='EDUCATION' draws the bars horizontally
ax = sns.countplot(data=df, y='EDUCATION', order=order)

# Write the count at the end of each bar
ax.bar_label(ax.containers[0])

plt.title('Education Level of Applicants')
plt.xlabel('Number of Applicants')
plt.ylabel('Education Level')
plt.show()

**Observation:** Most applicants have Secondary (1,031) or Higher education (426). The other levels are rare.

### Visualization 5 — Age vs Annual Income (Scatter Plot)
**Why:** A scatter plot shows whether two numerical columns are related.

In [ ]:
plt.figure(figsize=(9, 5))

# hue='Status' -> colour the dots by Approved / Rejected
# alpha=0.6    -> make the dots slightly see-through so overlapping dots are visible
sns.scatterplot(data=df, x='Age', y='Annual_income', hue='Status', alpha=0.6)

plt.title('Age vs Annual Income')
plt.xlabel('Age (years)')
plt.ylabel('Annual Income')
plt.show()

**Observation:** There is no clear trend between age and income, and approved and rejected dots are mixed together.

### Visualization 6 — Correlation Heatmap
**Why:** A heatmap compares many numerical columns at once. Values near +1 or −1 mean a strong relationship, and values near 0 mean none. (`Mobile_phone` is left out because it is always 1.)

In [ ]:
# Columns to compare
heatmap_cols = ['CHILDREN', 'Family_Members', 'Annual_income', 'Age', 'label']

# corr() calculates the correlation between every pair of these columns
corr = df[heatmap_cols].corr()

plt.figure(figsize=(8, 6))

# annot=True -> write the value inside each box
# fmt='.2f'  -> show 2 decimal places
# cmap='coolwarm' -> blue for negative, red for positive
# vmin/vmax  -> fix the colour scale from -1 to +1
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)

plt.title('Correlation Heatmap')
plt.show()

**Observation:** `CHILDREN` and `Family_Members` are strongly related (0.89). `label` is close to 0 with every column, so no single column decides approval.

## Task 7 — Important Insights

1. 1,548 applicants and no duplicate rows. Missing values were filled, so no rows were lost.
2. 88.7% of applications were approved, so the data is **imbalanced**.
3. Income is **right-skewed**, with 73 high-income outliers.
4. Most applicants have Secondary or Higher education.
5. Age and income show no clear relationship.
6. `CHILDREN` and `Family_Members` are strongly related, and no single column decides approval.

## Conclusion

Choosing a chart that matches the type of data (count plots for categories; histograms, box plots, scatter plots, and heatmaps for numbers) made the patterns in this dataset quick and easy to see: the approval imbalance, the skewed income with outliers, and the lack of a single deciding factor.